# 01. Exploración inicial

**Objetivo**: abrir los 7 datasets y entender qué hay en cada uno (forma, tipos de dato, nulos, duplicados, valores únicos) — sin corregir nada todavía. 

In [ ]:
# Cargamos las librerias
import pandas as pd
import os

In [36]:
print(os.getcwd()) # Estote imprime la ruta completa de la carpeta desde la que Python está "mirando" ahora mismo

c:\Users\paula\PROYECTOS\playloop-game-industry-business-case\notebooks


In [37]:
ficheros = ['usuarios','sesiones','partidas','compras','ingresos_ads','gasto_ua','productos_iap']
ruta = '../data/raw/' # Ruta relativa
dfs = {} # Diccionario para guardar los DataFrame

print("Dimensiones de los datasets:")
# Creamos un bucle para guardar cada DataFrame usando como clave el valor que tiene f dentro de la vuelta
for f in ficheros:
    dfs[f] = pd.read_csv(f"{ruta}{f}.csv")
    print(f"{f}: {dfs[f].shape}")

Dimensiones de los datasets:
usuarios: (8040, 6)
sesiones: (44945, 5)
partidas: (61716, 7)
compras: (4558, 8)
ingresos_ads: (19999, 6)
gasto_ua: (280, 6)
productos_iap: (14, 5)


**Granularidad de cada tabla**
- **usuarios** → 1 fila = 1 jugador
- **sesiones** → 1 fila = 1 apertura de la app
- **partidas** → 1 fila = 1 nivel jugado
- **compras** → 1 fila = 1 transacción de compra (IAP)
- **ingresos_ads** → 1 fila = 1 visualización de anuncio
- **gasto_ua** → 1 fila = 1 canal en 1 día (agregado, no por jugador)
- **productos_iap** → 1 fila = 1 producto del catálogo

In [38]:
# Nombres de columns
print("Listado de columnas en cada tabla")
for nombre, df in dfs.items():
    print(f"---{nombre}---")
    print(list(df.columns))

Listado de columnas en cada tabla
---usuarios---
['user_id', 'fecha_instalacion', 'canal_adquisicion', 'pais', 'plataforma', 'version_app']
---sesiones---
['evento_id', 'user_id', 'fecha', 'duracion_segundos', 'plataforma']
---partidas---
['partida_id', 'user_id', 'nivel', 'fecha', 'resultado', 'movimientos', 'boosters_usados']
---compras---
['compra_id', 'user_id', 'producto_id', 'fecha', 'importe', 'moneda', 'plataforma', 'reembolso']
---ingresos_ads---
['id', 'user_id', 'fecha', 'tipo_ad', 'red_ad', 'ingreso']
---gasto_ua---
['canal', 'fecha', 'gasto', 'instalaciones', 'impresiones', 'clics']
---productos_iap---
['producto_id', 'nombre', 'tipo', 'precio_base', 'monedas']


### Exploracion de usuarios

In [39]:
usuarios =  dfs['usuarios']

# Tipo de datos en las columnas
print("--- Tipos de datos ---")
print(usuarios.dtypes)

# Primeras filas
print("\n--- Encabezado ---")
print(usuarios.head())

# Nulos por columna
print("\n--- Nulos por columna ---")
print(usuarios.isnull().sum())


# Analizamos los usuarios y vemos si hay duplicados
print("\n--- Analisis de nº usuarios ---")
print(f"Filas:{len(usuarios)}")
print(f"Usuarios únicos: {usuarios['user_id'].nunique()}")
print(f"Duplicados: {len(usuarios)-usuarios['user_id'].nunique()}")

--- Tipos de datos ---
user_id              int64
fecha_instalacion      str
canal_adquisicion      str
pais                   str
plataforma             str
version_app            str
dtype: object

--- Encabezado ---
   user_id fecha_instalacion canal_adquisicion    pais plataforma version_app
0     1474        9 jul 2024          coinrush      es    Android       3.0.1
1     2148        12/07/2024          CoinRush      es    android         NaN
2     5036        2024-06-19            google  México        ios       2.9.5
3     1108        2024-07-03          coinrush      ES    Android       3.3.0
4     1931        2024-07-01          Coinrush  mexico        ios       3.1.0

--- Nulos por columna ---
user_id                0
fecha_instalacion      0
canal_adquisicion      0
pais                   0
plataforma             0
version_app          297
dtype: int64

--- Analisis de nº usuarios ---
Filas:8040
Usuarios únicos: 8000
Duplicados: 40


In [40]:
# Vamos a ver si los duplicados son copias exactas (toda la columna) o el mismo used_i con datos distintos
print("Filas duplicadas exactas (todas las columnas iguales):",usuarios.duplicated().sum())
print("user_id repetido (resto columnas cambia):",usuarios.duplicated(subset='user_id').sum())

duplicados_usuario = usuarios[usuarios.duplicated(subset='user_id',keep=False)]
duplicados_usuario.sort_values('user_id')


Filas duplicadas exactas (todas las columnas iguales): 1
user_id repetido (resto columnas cambia): 40


,user_id,fecha_instalacion,canal_adquisicion,pais,plataforma,version_app
631,32,2024-06-09,CoinRush,ES,ios,3.3.0
3690,32,2024-06-09,coinrush,ES,ios,3.3.0
4237,185,15/07/2024,CoinRush,es,iOS,3.1.0
4428,185,15/07/2024,coinrush,ES,ios,3.1.0
1222,473,21/07/2024,Coinrush,Italia,iOS,3.2.1
...,...,...,...,...,...,...
1496,7310,2024-06-30,TikTok,IT,iOS,3.2.1
224,7643,2024-07-11,organico,ES,ios,NaN
2947,7643,2024-07-11,organic,ES,iOS,NaN
1490,7975,2024-06-21,organico,MX,ios,3.0.1


In [135]:
# Valores unicos de canal_adquisicion
usuarios['canal_adquisicion'].unique()

<StringArray>
[    'coinrush',     'CoinRush',       'google',     'Coinrush',
         'meta',     'facebook',       'TikTok',     'orgánico',
    'coin_rush',      'tik_tok',      'organic',       'Google',
   'Google Ads',       'tiktok',     'organico',         'Meta',
   'google_ads', 'Apple Search',     'Facebook', 'apple_search',
          'ASA', 'apple search',     'Organico']
Length: 23, dtype: str

### Exploración — usuarios.csv

- `fecha_instalacion` está como texto (`str`), no como fecha. Las fechas no están estandarizadas: al menos 3 formatos distintos conviven 
- `canal_adquisicion` no está estandarizado: mismo canal varias formas (`coinrush`, `CoinRush`, `Coinrush`...).
- `pais` no está estandarizado: mayúsculas/minúsculas mezcladas, variantes con/sin acento y código vs. nombre completo (`es`, `ES`, `México`, `mexico`, `MX`).
- `plataforma` no está estandarizada: mayúsculas/minúsculas mezcladas (`Android`, `android`, `ios`, `iOS`).
- `version_app` tiene 297 nulos.
- **Duplicados de `user_id`**: 8.040 filas para solo 8.000 `user_id` únicos (40 filas de más), pero solo 1 es una copia exacta fila a fila. Al revisar las 40 parejas (`duplicated(subset='user_id', keep=False)`), el patrón es siempre el mismo: **misma fecha de instalación y misma versión de app dentro de cada pareja**, y lo único que cambia es cómo está escrito el mismo valor categórico (`CoinRush`/`coinrush`, `organico`/`organic`, `MX`/`mexico`...).
  - Hipótesis: no parece un reintento técnico real (que daría una fila idéntica), sino dos fuentes de datos distintas (BigQuery, AppsFlyer, tiendas) registrando el mismo evento con su propia convención de texto, mezcladas en esta exportación.
  - **Implicación para la limpieza**: estandarizar primero el texto de las columnas categóricas y *solo después* volver a buscar duplicados — así, casos como `CoinRush`/`coinrush` se vuelven filas idénticas de verdad y son fáciles de eliminar sin perder información.

### Exploracion de sesiones

In [46]:
sesiones = dfs['sesiones']

# Tipo de datos en las columnas 
print("--- Tipos de datos ---")
print(sesiones.dtypes)

# Primeras filas 
print("\n--- Encabezado ---")
print(sesiones.head())

# Nulos por columnas
print("\n--- Nulos por columna ---")
print(sesiones.isnull().sum())

# Análisis de duplicados
sesiones_totales = len(sesiones)
print("\n--- Análisis de duplicados ---")
print(f"Filas: {sesiones_totales}")
print(f"Sesiones unicas: {sesiones['evento_id'].nunique()}")
print(f"Duplicados por evento_id: {sesiones_totales-sesiones['evento_id'].nunique()}")

--- Tipos de datos ---
evento_id            int64
user_id              int64
fecha                  str
duracion_segundos    int64
plataforma             str
dtype: object

--- Encabezado ---
   evento_id  user_id        fecha  duracion_segundos plataforma
0   10020515     4475  14 jul 2024                263    Android
1   10017007     4009   2024-06-27                229        IOS
2   10003905     1363   2024-06-21                210        iOS
3   10042023     7630   11/07/2024                103        ios
4   10018908     4254   07/06/2024                332    Android

--- Nulos por columna ---
evento_id            0
user_id              0
fecha                0
duracion_segundos    0
plataforma           0
dtype: int64

--- Análisis de duplicados ---
Filas: 44945
Sesiones unicas: 44633
Duplicados por evento_id: 312


In [42]:
# Duplicados exactos o de evento_id
print("Filas duplicadas exactas (todas las columnas iguales):",sesiones.duplicated().sum())
print("Sesiones duplicadas (resto columnas pueden cambiar):",sesiones.duplicated(subset='evento_id').sum())

Filas duplicadas exactas (todas las columnas iguales): 312
Sesiones duplicadas (resto columnas pueden cambiar): 312


In [43]:
# Cruzamos las tablas para ver si las claves casan
cruce = sesiones.merge(usuarios[['user_id']].drop_duplicates(),on='user_id',how='left',indicator=True)

print(f"Sesiones: {sesiones_totales}")
print(f"Sesiones con usario valido: {(cruce['_merge']=='both').sum()}")
print(f"Sesiones huerfanas: {(cruce['_merge']=='left_only').sum()}")

Sesiones: 44945
Sesiones con usario valido: 44945
Sesiones huerfanas: 0


In [137]:
# Analisis de las sesiones 
print(sesiones['duracion_segundos'].describe())

print(f"Duracion minima (cruda): {sesiones['duracion_segundos'].min()}")
print(f"Duracion maxima (cruda): {sesiones['duracion_segundos'].max()}")

malas = ((sesiones['duracion_segundos'] <= 0) | (sesiones['duracion_segundos'] > 3600))
print(f"Sesiones fuera de rango: {malas.sum()}")

count     44945.000000
mean        583.905039
std       20005.028244
min         -30.000000
25%         104.000000
50%         180.000000
75%         255.000000
max      999999.000000
Name: duracion_segundos, dtype: float64
Duracion minima (cruda): -30
Duracion maxima (cruda): 999999
Sesiones fuera de rango: 60


### Exploración — sesiones.csv

- `fecha` está como texto (`str`), no como fecha, y no está estandarizada: varios formatos distintos conviven (igual que en `usuarios.csv`).
- `plataforma` no está estandarizada: mayúsculas/minúsculas mezcladas.
- No hay nulos en ninguna columna.
- `sesiones` duraciones extrañas
- **Duplicados**: 312 `evento_id` repetidos, y los 312 son copias exactas fila a fila (`duplicated()` y `duplicated(subset='evento_id')` coinciden). A diferencia de `usuarios.csv`, aquí sí encaja con la explicación del caso: reintentos técnicos del sistema del juego. Limpieza sencilla: `drop_duplicates()` sin pasos previos de estandarización de texto.
- **Integridad referencial**: las 44.945 sesiones tienen un `user_id` válido en `usuarios` — 0 sesiones huérfanas.

### Exploración de partidas


In [60]:
partidas = dfs['partidas']


# Tipo de datos en las columnas 
print("--- Tipos de datos ---")
print(partidas.dtypes)

# Primeras filas 
print("\n--- Encabezado ---")
print(partidas.head())

# Nulos por columnas
print("\n--- Nulos por columna ---")
print(partidas.isnull().sum())


# Análisis de duplicados
partidas_totales = len(partidas)
print("\n--- Análisis de duplicados ---")
print(f"Filas: {partidas_totales}")
print(f"Partidas unicas: {partidas['partida_id'].nunique()}")
print(f"Duplicados por partida_id: {partidas_totales - partidas['partida_id'].nunique()}")


--- Tipos de datos ---
partida_id           int64
user_id              int64
nivel                int64
fecha                  str
resultado              str
movimientos          int64
boosters_usados    float64
dtype: object

--- Encabezado ---
   partida_id  user_id  nivel       fecha resultado  movimientos  \
0    20051872     7029      6  2024-07-11      fail           12   
1    20028138     4696      3  2024-06-11      fail           24   
2    20053463     7224      3  2024-07-04       win           12   
3    20051328     6956      4  2024-07-04    ganada           14   
4    20016282     3492      3  2024-07-07       win           32   

   boosters_usados  
0              0.0  
1              0.0  
2              1.0  
3              2.0  
4              0.0  

--- Nulos por columna ---
partida_id            0
user_id               0
nivel                 0
fecha                 0
resultado             0
movimientos           0
boosters_usados    1854
dtype: int64

--- Anális

In [65]:
# Vemos los diferentes formatos de fecha para ver si hay que corregir
# No se aprecian en el .head()
sesiones['fecha'].unique()

<StringArray>
['14 jul 2024',  '2024-06-27',  '2024-06-21',  '11/07/2024',  '07/06/2024',
  '30/05/2024',  '12/07/2024',  '2024-07-03',  '10/07/2024', '15 jun 2024',
 ...
 '29 may 2024',  '2024-05-28',  '7 jun 2024',  '01/06/2024',  '5 jun 2024',
  '2024-06-05', '19 jun 2024',  '4 jun 2024', '28 may 2024',  '2 jun 2024']
Length: 168, dtype: str

In [75]:
# Vemos integridad referencial entre tablas
cruce_partidas = partidas.merge(usuarios[['user_id']].drop_duplicates(),on='user_id',how='left',indicator=True)

print(f"Partidas totales: {partidas_totales}")
print(f"Partidas con usuario valido: {(cruce_partidas['_merge']=='both').sum()}")
print(f"Partidas con sesiones huerfanas: {(cruce_partidas['_merge']=='left').sum()}")

Partidas totales: 61716
Partidas con usuario valido: 61716
Partidas con sesiones huerfanas: 0


### Exploración — partidas.csv

- `fecha` está como texto (`str`), no como fecha, y no está estandarizada: hay varios formatos distintos mezclados (mismo patrón que en `usuarios.csv` y `sesiones.csv`).
- `resultado` no está estandarizado: mezcla valores en inglés y español (`fail`, `win`, `ganada`) — **pendiente**: revisar `.unique()` completo para ver si hay más variantes (ej. una versión en español de "fail").
- `boosters_usados` tiene 1.854 nulos. Importante: **0 y nulo no son lo mismo aquí** — un 0 es un dato real ("se registró que no usó ningún booster"), un nulo podría significar "no se registró este dato". **Pendiente decidir** cómo tratar cada caso en la fase de limpieza.
- **Duplicados**: 307 `partida_id` repetidos (61.716 filas vs. 61.409 únicas) — **pendiente**: confirmar si son copias exactas (como en `sesiones`) o el mismo `partida_id` con datos distintos (como en `usuarios`), con el mismo chequeo de `duplicated()` completo vs. `duplicated(subset='partida_id')`.
- **Integridad referencial con `usuarios`**: las 61.716 partidas tienen un `user_id` válido — 0 huérfanas.

### Exploracion de compras


In [76]:
compras = dfs['compras']

In [82]:
# Tipo de datos en las columnas 
print("--- Tipos de datos ---")
print(compras.dtypes)

# Primeras filas 
print("\n--- Encabezado ---")
print(compras.head())

# Nulos por columnas
print("\n--- Nulos por columna ---")
print(compras.isnull().sum())

# Análisis de duplicados
compras_totales = len(compras)
print("\n--- Análisis de duplicados ---")
print(f"Filas: {compras_totales}")
print(f"Partidas unicas: {compras['compra_id'].nunique()}")
print(f"Duplicados por partida_id: {compras_totales - compras['compra_id'].nunique()}")

--- Tipos de datos ---
compra_id      int64
user_id        int64
producto_id    int64
fecha            str
importe          str
moneda           str
plataforma       str
reembolso      int64
dtype: object

--- Encabezado ---
   compra_id  user_id  producto_id       fecha importe moneda plataforma  \
0   30004059     7541            3  6 jun 2024  4,99 €    eur    Android   
1   30003960     6828            1  25/06/2024  0,99 €    eur        ios   
2   30000822     3481            1  2024-07-03  0,99 €    eur        iOS   
3   30003149     5905           10  2024-06-14  1,99 €    EUR    Android   
4   30002203     5013           12  2024-07-01  2,49 €    EUR    android   

   reembolso  
0          0  
1          0  
2          0  
3          0  
4          0  

--- Nulos por columna ---
compra_id      0
user_id        0
producto_id    0
fecha          0
importe        0
moneda         0
plataforma     0
reembolso      0
dtype: int64

--- Análisis de duplicados ---
Filas: 4558
Partidas

In [88]:
# Vemos que valores hay en compras en el importe
compras['importe'].unique()

<StringArray>
[    '4,99 €',     '0,99 €',     '1,99 €',     '2,49 €',     '9,99 €',
     '2,99 €',     '3,99 €',    '19,99 €',    '-0,99 €',    '29,99 €',
    '-3,99 €',    '-2,49 €',  '19,30 MXN',    '-1,99 €',    '49,99 €',
  '77,81 MXN',    '-4,99 €',  '58,31 MXN',    '-2,99 €', '974,81 MXN',
   '-19,99 €',  '97,31 MXN']
Length: 22, dtype: str

In [90]:
# Comprobar si el importe negativo esta relacionado con reembolso <> 1
negativos_sin_marcar = compras[(compras['importe'].str.contains('-')) & (compras['reembolso'] == 0)]
print("Importes negativos sin marcar como reembolso:", len(negativos_sin_marcar))

Importes negativos sin marcar como reembolso: 0


In [93]:
# Integridad tablas
cruce_compras = compras.merge(usuarios[['user_id']].drop_duplicates(),on='user_id',how='left',indicator=True)

print(f'Compras totales: {compras_totales}')
print(f"Compras con usuario valido: {(cruce_compras['_merge']=='both').sum()}")
print(f"Compras huerfanas: {(cruce_compras['_merge']=='left').sum()}")

Compras totales: 4558
Compras con usuario valido: 4558
Compras huerfanas: 0


### Exploración — compras.csv

- `fecha` está como texto (`str`), no como fecha, y no está estandarizada: varios formatos distintos, igual que en el resto de tablas.
- `moneda` no está estandarizada: mismo valor escrito de formas distintas (`eur`, `EUR`, `€`) — y además hay transacciones en una divisa real distinta, `MXN` (pesos mexicanos), no solo un problema de formato de texto.
- `plataforma` no está estandarizada: mayúsculas/minúsculas mezcladas.
- No hay duplicados.
- `importe` se comporta como texto porque incluye el símbolo `€` (o el sufijo `MXN`) y usa coma como separador decimal — habrá que limpiar el texto y convertir a número en la fase de limpieza.
- No hay nulos.
- `importe` tiene valores negativos (reembolsos). **Comprobado**: los 0 casos de importe negativo sin `reembolso == 1` — la columna `reembolso` es consistente con el signo del importe.
- **Integridad referencial con `usuarios`**: las 4.558 compras tienen un `user_id` válido — 0 huérfanas.
- **Pendiente para el cálculo de KPIs**: las compras en MXN deben convertirse a EUR con un tipo de cambio antes de sumar ingresos entre canales — mezclar ambas divisas sin convertir falsearía el LTV. 

### Exploracion productos iap

In [109]:
productos_iap = dfs['productos_iap']

In [119]:
print("--- Tipos de datos ---")
print(productos_iap.dtypes)

print("\n--- Encabezado ---")
print(productos_iap.head(14)) # solo son 14 filas

print("\n--- Nulos por columna ---")
print(productos_iap.isnull().sum())

# Análisis de duplicados
productos_iap_totales = len(productos_iap)
print("\n--- Análisis de duplicados ---")
print(f"Filas: {productos_iap_totales}")
print(f"Partidas unicas: {productos_iap['producto_id'].nunique()}")
print(f"Duplicados por producto_id: {productos_iap_totales - productos_iap['producto_id'].nunique()}")

--- Tipos de datos ---
producto_id      int64
nombre             str
tipo               str
precio_base        str
monedas        float64
dtype: object

--- Encabezado ---
    producto_id               nombre             tipo precio_base  monedas
0             1       Pack Monedas S          Monedas      0,99 €    100.0
1             2       Pack Monedas M          monedas      2,99 €    350.0
2             3       Pack Monedas L          monedas      4,99 €    650.0
3             4      Pack Monedas XL          monedas      9,99 €   1500.0
4             5     Pack Monedas XXL          monedas     19,99 €   3500.0
5             6    Pack Monedas Mega          Monedas     49,99 €  10000.0
6             7      Quitar Anuncios  quitar_anuncios      3,99 €      NaN
7             8       Pase Temporada      suscripcion      7,99 €      NaN
8             9  Pase Temporada Plus      suscripcion     14,99 €      NaN
9            10           Pack Vidas          monedas      1,99 €     50.0
10 

In [123]:
# Integridad de compras a productos
cruce_productos = compras.merge(productos_iap[['producto_id']],on='producto_id',how='left',indicator=True)

print(f"Compras totales: {compras_totales}")
print(f"Compras con producto valido: {(cruce_productos['_merge'] == 'both').sum()}")
print(f"Compras con producto huerfano: {(cruce_productos['_merge'] == 'left_only').sum()}")


Compras totales: 4558
Compras con producto valido: 4558
Compras con producto huerfano: 0


### Exploración — productos_iap.csv

- `precio_base` se comporta como texto porque incluye el símbolo `€` y usa coma decimal — misma limpieza pendiente que en `compras.importe`.
- `tipo` no está estandarizado: `Monedas`/`monedas` con mayúsculas mezcladas, más dos categorías distintas legítimas (`quitar_anuncios`, `suscripcion`).
- `nombre` tiene inconsistencias de mayúsculas/espacios.
- `monedas` tiene 3 nulos, pero **no es un problema de calidad**: coinciden exactamente con los productos de tipo `quitar_anuncios` y `suscripcion`, que no tienen sentido que lleven una cantidad de monedas asociada — es una ausencia estructural esperada, no un dato perdido.
- No hay duplicados.
- **Integridad referencial con `compras`**: las 4.558 compras tienen un `producto_id` válido en `productos_iap` — 0 huérfanas.

### Exploración de ingresos ads

In [94]:
ingresos_ads = dfs['ingresos_ads']

In [95]:
ingresos_ads.head()

,id,user_id,fecha,tipo_ad,red_ad,ingreso
0,40002095,2403,2024-07-06,Interstitial,admob,0.0797
1,40006648,2857,06/07/2024,rewarded,AdMob,0.0531
2,40010934,4805,2024-07-17,Rewarded,admob,0.0348
3,40015469,6446,03/07/2024,Rewarded,admob,0.0183
4,40016160,6679,2024-07-16,recompensado,ironsource,0.4685


In [101]:
# Tipo de datos en las columnas 
print("--- Tipos de datos ---")
print(ingresos_ads.dtypes)
# Primeras filas 
print("\n--- Encabezado ---")
print(ingresos_ads.head())

# Nulos por columnas
print("\n--- Nulos por columna ---")
print(ingresos_ads.isnull().sum())

# Análisis de duplicados
ingresos_ads_totales= len(ingresos_ads)
print("\n--- Análisis de duplicados ---")
print(f"Filas: {ingresos_ads_totales}")
print(f"Partidas unicas: {ingresos_ads['id'].nunique()}")
print(f"Duplicados por partida_id: {ingresos_ads_totales - ingresos_ads['id'].nunique()}")

--- Tipos de datos ---
id           int64
user_id      int64
fecha          str
tipo_ad        str
red_ad         str
ingreso    float64
dtype: object

--- Encabezado ---
         id  user_id       fecha       tipo_ad      red_ad  ingreso
0  40002095     2403  2024-07-06  Interstitial       admob   0.0797
1  40006648     2857  06/07/2024      rewarded       AdMob   0.0531
2  40010934     4805  2024-07-17      Rewarded       admob   0.0348
3  40015469     6446  03/07/2024      Rewarded       admob   0.0183
4  40016160     6679  2024-07-16  recompensado  ironsource   0.4685

--- Nulos por columna ---
id           0
user_id      0
fecha        0
tipo_ad      0
red_ad       0
ingreso    600
dtype: int64

--- Análisis de duplicados ---
Filas: 19999
Partidas unicas: 19999
Duplicados por partida_id: 0


In [105]:
# Integridad tablas
cruce_ingresos_ads = ingresos_ads.merge(usuarios[['user_id']].drop_duplicates(),on='user_id',how='left',indicator=True)

print(f"Ingresos_ads totales: {ingresos_ads_totales}")
print(f"Con usuario valido: {(cruce_ingresos_ads['_merge'] == 'both').sum()}")
print(f"Huerfanas: {(cruce_ingresos_ads['_merge'] == 'left_only').sum()}")

Ingresos_ads totales: 19999
Con usuario valido: 19999
Huerfanas: 0


### Exploración — ingresos_ads.csv

- `fecha` está como texto (`str`), no como fecha, y no está estandarizada: varios formatos distintos, igual que en el resto de tablas.
- `tipo_ad` no está estandarizado: mezcla valores en inglés y español, aunque al menos todos en minúsculas.
- `red_ad` (la red de publicidad/mediador que sirvió el anuncio) tampoco está estandarizada.
- `ingreso` tiene 600 nulos — pendiente decidir en la limpieza si significa "no generó ingreso" o "no se registró el dato".
- No hay duplicados.
- **Integridad referencial con `usuarios`**: todas las filas tienen un `user_id` válido — 0 huérfanas.

### Exploracion gasto ua

In [124]:
gasto_ua = dfs['gasto_ua']

In [132]:
print("--- Tipos de datos ---")
print(gasto_ua.dtypes)

print("\n--- Encabezado ---")
print(gasto_ua.head())

print("\n--- Nulos por columna ---")
print(gasto_ua.isnull().sum())

print("\n--- Duplicados ---")
print(gasto_ua.duplicated().sum())

--- Tipos de datos ---
canal                str
fecha                str
gasto                str
instalaciones      int64
impresiones        int64
clics            float64
dtype: object

--- Encabezado ---
          canal       fecha    gasto  instalaciones  impresiones  clics
0  apple search  2024-06-22  11,53 €             11         1294   29.0
1      Facebook  12/07/2024  20,60 €             20         2307   67.0
2       organic  16/07/2024   0,00 €              2          249    5.0
3     coin_rush  2024-07-19  32,05 €             72         4997  146.0
4      coinrush  2024-07-02  58,45 €            129        15381  481.0

--- Nulos por columna ---
canal             0
fecha             0
gasto             0
instalaciones     0
impresiones       0
clics            11
dtype: int64

--- Duplicados ---
0


In [130]:
# Valores unicos de canal
gasto_ua['canal'].unique()

<StringArray>
['apple search',     'Facebook',      'organic',    'coin_rush',
     'coinrush',       'google',       'TikTok',     'orgánico',
         'meta',     'Coinrush',      'tik_tok',       'tiktok',
         'Meta',   'Google Ads',     'organico',          'ASA',
 'apple_search',     'Organico',   'google_ads',     'CoinRush',
 'Apple Search',       'Google',     'facebook']
Length: 23, dtype: str

## Exploración — gasto_ua.csv

- `fecha` está como texto (`str`) y no está estandarizada: varios formatos distintos, igual que en el resto de tablas.
- `gasto` se comporta como texto por el símbolo `€` y la coma decimal — además, algunas filas usan el punto como separador de miles (ej. `1.263,71 €`), lo que exige limpiar en el orden correcto (quitar puntos de miles antes de convertir la coma en separador decimal).
- `clics` tiene 11 nulos. Por eso pandas la tipó como `float64` en vez de `int64` (no se puede tener un `NaN` en una columna de enteros puros) — se podrá volver a `int` tras decidir cómo rellenar esos nulos.
- `canal` no está estandarizado: 23 formas de texto distintas para solo 6 canales reales (`CoinRush`, `Apple Search`/`ASA`, `Google`, `TikTok`, `Meta`/`Facebook`, `Orgánico`). **Pregunta abierta**: ¿"Meta" y "Facebook" son el mismo canal publicitario o dos canales distintos que casualmente se agruparían al estandarizar el texto? Requiere una decisión de negocio, no solo limpieza de texto.
- No hay duplicados.